In [3]:
import pandas as pd
from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "sqlAnkita%230987"
host = "localhost"
port = 3306
database = "lending_portfolio"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Load the ML view
query = """
SELECT *
FROM finrisk_ml_dataset;
"""

In [2]:
df = pd.read_sql(query, engine)

print("Dataset loaded successfully")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset loaded successfully
Shape: (1303638, 73)

First 5 rows:
   loan_id  borrower_id  loan_amnt  funded_amnt  funded_amnt_inv  int_rate  \
0  1000000      5000000      30000        30000          30000.0     22.35   
1  1000001      5000001      40000        40000          40000.0     16.14   
2  1000002      5000002      20000        20000          20000.0      7.56   
3  1000003      5000003       4500         4500           4500.0     11.31   
4  1000004      5000004       8425         8425           8425.0     27.27   

   installment  policy_code  delinq_amnt  target  ...  num_tl_op_past_12m  \
0      1151.16            1          0.0       0  ...                 2.0   
1       975.71            1          0.0       0  ...                 4.0   
2       622.68            1          0.0       0  ...                 1.0   
3       147.99            1          0.0       0  ...                 4.0   
4       345.18            1          0.0       0  ...                 2.0   

   p

In [4]:
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))
print("\nTarget distribution:")
print(df["target"].value_counts())


Missing values:
dti                      2
borrower_id              0
loan_amnt                0
funded_amnt              0
funded_amnt_inv          0
int_rate                 0
installment              0
policy_code              0
loan_id                  0
delinq_amnt              0
target                   0
annual_inc               0
mths_since_rcnt_il       0
mths_since_recent_bc     0
mths_since_recent_inq    0
delinq_2yrs              0
inq_last_6mths           0
open_acc                 0
pub_rec                  0
revol_bal                0
dtype: int64

Target distribution:
target
0    1041952
1     261686
Name: count, dtype: int64


In [5]:
print("\nTarget percentage:")
print(
    df["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Target percentage:
target
0    79.93
1    20.07
Name: proportion, dtype: float64


In [6]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate loan IDs:", df["loan_id"].duplicated().sum())

Duplicate rows: 0
Duplicate loan IDs: 0


In [7]:
X = df.drop(columns=["target"])
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1303638, 72)
y shape: (1303638,)


In [8]:
print("\nFeature information:")
print(X.info())

print("\nFeature data types:")
print(X.dtypes.value_counts())

print("\nMissing values:")
missing_values = X.isnull().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

print("\nUnique values per column:")
unique_counts = X.nunique().sort_values()
print(unique_counts.head(20))

print("\nConstant columns:")
constant_columns = [
    column for column in X.columns
    if X[column].nunique(dropna=False) <= 1
]
print(constant_columns)


Feature information:
<class 'pandas.DataFrame'>
RangeIndex: 1303638 entries, 0 to 1303637
Data columns (total 72 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   loan_id                     1303638 non-null  int64  
 1   borrower_id                 1303638 non-null  int64  
 2   loan_amnt                   1303638 non-null  int64  
 3   funded_amnt                 1303638 non-null  int64  
 4   funded_amnt_inv             1303638 non-null  float64
 5   int_rate                    1303638 non-null  float64
 6   installment                 1303638 non-null  float64
 7   policy_code                 1303638 non-null  int64  
 8   delinq_amnt                 1303638 non-null  float64
 9   annual_inc                  1303638 non-null  float64
 10  dti                         1303636 non-null  float64
 11  mths_since_rcnt_il          1303638 non-null  float64
 12  mths_since_recent_bc        1303638 non-null 

In [9]:
print("\nID columns:")
print([
    column for column in X.columns
    if "id" in column.lower()
])


ID columns:
['loan_id', 'borrower_id']


In [10]:
X = X.drop(columns=["loan_id", "borrower_id"])

print("\nX shape after removing IDs:", X.shape)


X shape after removing IDs: (1303638, 70)


In [11]:
print("\nAll feature names:")
for column in X.columns:
    print(column)


All feature names:
loan_amnt
funded_amnt
funded_amnt_inv
int_rate
installment
policy_code
delinq_amnt
annual_inc
dti
mths_since_rcnt_il
mths_since_recent_bc
mths_since_recent_inq
delinq_2yrs
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc
collections_12_mths_ex_med
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_within_12_mths
mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
mort_acc
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats
num_tl_120dpd_2m
num_tl_30dpd
num_tl_90g_dpd_24m
num_tl_op_past_12m
pct_tl_nvr_dlq
percent_bc_gt_75
pub_rec_bankruptcies
tax_liens
tot_hi_cred_lim
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit


In [ ]:
import pandas as pd

df = pd.read_csv("C:\\Users\\HP\\Documents\\finrisk\\notebooks\\finrisk_ml_dataset.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head(3).to_string())
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(20))

In [28]:
df["target"].value_counts()

target
0    1041952
1     261686
Name: count, dtype: int64

In [34]:
for column in df.columns:
    if df[column].isnull().sum() > 0:
        print(f"Column: {column} has {df[column].isnull().sum()} missing values")

Column: dti has 2 missing values


In [37]:
df['dti'] = df['dti'].fillna(df['dti'].median())

In [38]:
for column in df.columns:
    if column == "dti":
        print(f"Column: {column} has {df[column].isnull().sum()} missing values after filling with median")

Column: dti has 0 missing values after filling with median


In [42]:
df["dti"].isnull().sum()

np.int64(0)

In [46]:
import pandas as pd

file_path = r"C:\Users\HP\Documents\finrisk\data\raw\loan.csv"
chunk_size = 100_000

# 1. Read first chunk to extract column names and basic types
first_chunk = pd.read_csv(file_path, nrows=1000)
columns = first_chunk.columns.tolist()

# 2. Track metrics across the dataset
null_counts = {col: 0 for col in columns}
unique_sets = {col: set() for col in columns}
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
    total_rows += len(chunk)
    for col in columns:
        null_counts[col] += int(chunk[col].isna().sum())
        
        # Track up to 50 distinct samples to avoid memory blow-up
        if len(unique_sets[col]) < 50:
            valid_vals = chunk[col].dropna().unique()
            unique_sets[col].update(valid_vals[:10])

# 3. Build dictionary DataFrame
dictionary_rows = []
for col in columns:
    samples = list(unique_sets[col])[:3]
    null_cnt = null_counts[col]
    
    dictionary_rows.append({
        "Field Name": col,
        "Data Type": str(first_chunk[col].dtype),
        "Total Count": total_rows,
        "Null Count": null_cnt,
        "Null %": round((null_cnt / total_rows) * 100, 2) if total_rows > 0 else 0,
        "Sample Values": ", ".join(map(str, samples)),
        "Business Definition": "",
        "Allowed Values / Notes": ""
    })

dict_df = pd.DataFrame(dictionary_rows)
dict_df.to_csv("Data_Dictionary.csv", index=False)